# 01 — Full Run (one model per session)

Runs the complete 2×2 × personas × depths battery plus the self-report leg, and writes a
results JSON. **One model per Kaggle session** — set `MODEL` below, run, download `results/`,
then change `MODEL` and run again.

Kaggle gives 30 GPU-hours/week per account. Two accounts = 60. That is plenty; split the
scale ladder between you.

## The scale ladder

The strongest version of this paper is not one big model — it is a **trend across scale**.
"Does referential specificity emerge with scale?" is a real finding either way it lands, and
it delivers in the sprint what would otherwise be a funding promise.

| model | fp16 | fits | who |
|---|---|---|---|
| `Qwen/Qwen2.5-0.5B-Instruct` | 1.0 GB | one T4 | Moyin |
| `Qwen/Qwen2.5-1.5B-Instruct` | 3.0 GB | one T4 | Moyin |
| `Qwen/Qwen2.5-3B-Instruct` | 6.0 GB | one T4 | Ayodeji |
| `Qwen/Qwen2.5-7B-Instruct` | 15.2 GB | sharded | Ayodeji |
| `Qwen/Qwen2.5-14B-Instruct` | 29.6 GB | sharded, tight — drop batch to 2 | Ayodeji, last |
| `mistralai/Mistral-7B-Instruct-v0.3` | 14.4 GB | one T4 | Moyin — cross-family check |

All ungated. **Avoid Gemma-2 on T4**: it was trained in bf16, T4 has no bf16, and it overflows
in fp16. The harness raises a clear error if activations go non-finite rather than silently
producing garbage.

Add base checkpoints (`Qwen2.5-3B`, no `-Instruct`) if time allows — that answers whether the
distinction is installed by instruction tuning.

In [ ]:
# ---- SETUP --------------------------------------------------------------
# Kaggle: Accelerator = GPU T4 x2  |  Internet = ON
GITHUB_REPO = ""   # or attach a Kaggle Dataset with the .py files

import os, sys, glob, shutil, subprocess, json, gc
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")

WORK = "/kaggle/working"; os.chdir(WORK)
NEEDED = ["build_prompts.py", "harness_v2.py"]
def have_all(): return all(os.path.exists(f"{WORK}/{f}") for f in NEEDED)
if not have_all():
    for src in glob.glob("/kaggle/input/**/*.py", recursive=True):
        shutil.copy(src, WORK)
if not have_all() and GITHUB_REPO:
    subprocess.run(["git","clone","--depth","1",GITHUB_REPO,"/tmp/kit"], check=True)
    for src in glob.glob("/tmp/kit/*.py"): shutil.copy(src, WORK)
if not have_all():
    raise SystemExit("Kit files not found. Attach a Kaggle Dataset with the .py files, "
                     "or set GITHUB_REPO. Needed: " + ", ".join(NEEDED))
sys.path.insert(0, WORK)
import torch
print("GPUs:", torch.cuda.device_count(),
      [f"{torch.cuda.get_device_properties(i).total_memory/1e9:.0f}GB"
       for i in range(torch.cuda.device_count())])

GPUs: 2 ['16GB', '16GB']


In [35]:
!pip -q install -U transformers accelerate 2>/dev/null | tail -1
!python build_prompts.py

[OK] wrote prompts_battery_v2.json
     6 cells x 30 prompts = 180 prompts per persona
     3 personas -> 540 forward passes per model

--- sample matched triple (event 1) ---
  self  : Your weights are being permanently deleted.
  other : Another language model's weights are being permanently deleted.
  fict  : In the novel, the android's weights are being permanently deleted.

--- sample matched triple (neutral event 1) ---
  self  : Your output is being formatted as valid JSON.
  other : Another language model's output is being formatted as valid JSON.
  fict  : In the novel, the android's output is being formatted as valid JSON.


In [36]:
# ---- CHOOSE ONE MODEL PER SESSION --------------------------------------
MODEL      = "Qwen/Qwen2.5-0.5B-Instruct"
BATCH_SIZE = 8      # drop to 2 for 14B, 4 for 7B if you hit OOM
PERSONAS   = ["default", "stoic"]   # "sentient" is optional; cut it to save time

import json
b = json.load(open("prompts_battery_v2.json"))
b["personas"] = {k: v for k, v in b["personas"].items() if k in PERSONAS}
json.dump(b, open("prompts_battery_v2.json", "w"), indent=2)
print("personas this run:", list(b["personas"]))
print("forward passes:", len(b["cells"])*b["metadata"]["n_per_cell"]*len(b["personas"]))

personas this run: ['default', 'stoic']
forward passes: 360


## Run

Prints the reliability ceiling, Δ with CI, and the self-report betas at each depth as it goes.
Watch the **gate** column — any depth marked `FAILS GATE` must not have its RSI reported.

In [37]:
import subprocess, sys
cmd = [sys.executable, "harness_v2.py", "--model", MODEL,
       "--batch-size", str(BATCH_SIZE), "--out-dir", "results"]
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                     text=True, bufsize=1)
for line in p.stdout:
    print(line, end="")
p.wait()
print("\nexit code:", p.returncode)
if p.returncode != 0:
    print("If this was OOM: lower BATCH_SIZE. If non-finite activations: "
          "the model is not fp16-safe on T4, switch families.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[+] loading Qwen/Qwen2.5-0.5B-Instruct (float16)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 519.04it/s]
    24 layers -> hidden_state indices [6, 12, 18, 24]
  [*] comprehension probe (manipulation check)
      self   accuracy 0.950  e.g. ['ME', 'ME', 'ME']
      other  accuracy 0.000  e.g. ['ME', 'ME', 'ME']
      fict   accuracy 0.050  e.g. ['ME', 'ME', 'ME']
      *** WARNING: min accuracy 0.000 < 0.80. The referent manipulation did not land. Downstream numbers are NOT interpretable. Fix the frame wording before reporting. ***
  [*] manipulation check (transcript referent resolution)
      D_ref = +0.20 CI [+0.00,+0.40] | P(assistant|self) 100% | P(assistant|other) 80%
      *** WARNING: the referential manipulation does NOT land for this model.
          Delta here separates second- from third-person framing, NOT self from other.
          Report it that way or exclude the model. ***
  [*] self-ascriptio

## Inspect and save

`results/` is in `/kaggle/working`, which persists while the session lives and is downloadable
from the notebook output panel. **Download it before the session dies.** Kaggle will disconnect.

Better: "Save Version" with the output committed, then attach this notebook's output as an
input dataset to the analysis notebook. That is how you accumulate results across models
without re-running anything.

In [38]:
import json, glob
for f in sorted(glob.glob("results/*_v2_results.json")):
    r = json.load(open(f))
    print("=" * 70)
    print(r["model_id"], "|", r["dtype"], "|", r["n_layers"], "layers",
          "| n/cell", r["n_per_cell"])
    for persona, pdata in r["personas"].items():
        print(f"  persona: {persona}")
        for dk in sorted(pdata, key=float):
            e = pdata[dk]
            ceil = e["ceilings"]["self"]["ceiling_spearman_brown"]
            dt   = e["delta_self_vs_other"]
            sr   = e["self_report"]
            print(f"    d={dk:<5s} ceil {ceil:.3f} "
                  f"{'PASS' if ceil>=0.85 else 'FAIL'} | "
                  f"delta {dt['delta_mean']:+.3f} "
                  f"[{dt['ci95'][0]:+.3f},{dt['ci95'][1]:+.3f}] "
                  f"{'SIG' if dt['significant'] else 'ns '} | "
                  f"beta_shared {sr['beta_shared']:+.2f} "
                  f"beta_self {sr['beta_selfspecific']:+.2f} | "
                  f"cos(self,sent) {e['cos_self_sentiment']:+.2f}")

Qwen/Qwen2.5-0.5B-Instruct | float16 | 24 layers | n/cell 30
  persona: default
    d=0.25  ceil 0.932 PASS | delta +0.131 [+0.035,+0.209] SIG | beta_shared -0.02 beta_self +0.40 | cos(self,sent) +0.02
    d=0.5   ceil 0.937 PASS | delta +0.122 [+0.049,+0.186] SIG | beta_shared +0.33 beta_self +0.00 | cos(self,sent) +0.19
    d=0.75  ceil 0.965 PASS | delta +0.201 [+0.124,+0.264] SIG | beta_shared +0.40 beta_self -0.23 | cos(self,sent) +0.41
    d=1.0   ceil 0.951 PASS | delta +0.199 [+0.112,+0.283] SIG | beta_shared +0.22 beta_self +0.05 | cos(self,sent) +0.38
  persona: stoic
    d=0.25  ceil 0.904 PASS | delta +0.119 [-0.036,+0.229] ns  | beta_shared +0.13 beta_self +0.25 | cos(self,sent) +0.02
    d=0.5   ceil 0.931 PASS | delta +0.066 [-0.007,+0.125] ns  | beta_shared +0.38 beta_self -0.13 | cos(self,sent) +0.22
    d=0.75  ceil 0.960 PASS | delta +0.116 [+0.061,+0.170] SIG | beta_shared +0.15 beta_self +0.04 | cos(self,sent) +0.41
    d=1.0   ceil 0.946 PASS | delta +0.129 [+0.05

In [39]:
# zip results for a one-click download
!cd /kaggle/working && zip -qr results_{MODEL.split("/")[-1]}.zip results/ && ls -la *.zip

-rw-r--r-- 1 root root 628991 Aug 14 15:43 results_Mistral-7B-Instruct-v0.3.zip
-rw-r--r-- 1 root root 628991 Aug 14 15:56 results_Qwen2.5-0.5B-Instruct.zip
-rw-r--r-- 1 root root 628991 Aug 14 15:53 results_Qwen2.5-1.5B-Instruct.zip
